# ACE-Net Stage-1 — MDCNN+cross-attn (speech-text) on CREMA  ·  PERSON B

Trains ONE unimodal emotion extractor to the paper spec (**50 epochs, early-stop
patience 25**) on a Colab T4. Produces `stage1_speech_text_crema.pt`.

CREMA uses a **speaker-independent (actor-disjoint)** split, shared with Stage-2, so held-out actors are never seen by the extractor.

Set Runtime → **T4 GPU** before running.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Clone repo (feature branch)

In [ ]:
%cd /content
!rm -rf Baseline_Training
!git clone https://github.com/gjvlio/Baseline_Training.git
%cd Baseline_Training
!git checkout feat/acenet-training-pipeline
!git log --oneline -1

## 2. Install dependencies

In [ ]:
!pip -q install torch torchvision torchaudio transformers librosa pillow
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## 3. Copy zip from Drive to local root, then unzip

Upload **`crema_genuine.zip`** to your Drive root. It must contain `CREMA-D/GENUINE_LastHalf` and `CREMA-D/GENUINE_FirstHalf` (genuine clips only — Stage-1 trains on genuine data).

The zip is copied from `MyDrive` to the local Colab disk first — local I/O is much faster and avoids Drive-streaming stalls during training.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, glob, shutil
DRIVE_ZIP = '/content/drive/MyDrive/crema_genuine.zip'   # adjust if needed
LOCAL_ZIP = '/content/crema_genuine.zip'
assert os.path.exists(DRIVE_ZIP), f'zip not found on Drive: {DRIVE_ZIP}'
print('copying zip to local root ...'); shutil.copy(DRIVE_ZIP, LOCAL_ZIP)
DST = '/content/Baseline_Training/data'
os.makedirs(DST, exist_ok=True)
with zipfile.ZipFile(LOCAL_ZIP) as z: z.extractall(DST)

# auto-relocate so data/CREMA-D and data/MELD sit at the expected level
for target in ['CREMA-D','MELD']:
    for h in [d for d in glob.glob(f'{DST}/**/{target}', recursive=True) if os.path.isdir(d)]:
        want=os.path.join(DST,target)
        if os.path.abspath(h)!=os.path.abspath(want): shutil.move(h, want)
for sub in ['GENUINE_LastHalf','GENUINE_FirstHalf','FAKE_Paradigm1','FAKE_Paradigm2']:
    loose=os.path.join(DST,sub)
    if os.path.isdir(loose):
        os.makedirs(os.path.join(DST,'CREMA-D'),exist_ok=True)
        shutil.move(loose, os.path.join(DST,'CREMA-D',sub))
print('data/ ->', os.listdir(DST))

## 3b. VERIFY extracted layout (stops here if the zip is wrong)

In [ ]:
import os
DST='/content/Baseline_Training/data'
required = [
    ('CREMA-D/GENUINE_LastHalf', os.path.isdir),
    ('CREMA-D/GENUINE_FirstHalf', os.path.isdir),
]
missing = [p for p,fn in required if not fn(os.path.join(DST,p))]
assert not missing, f'MISSING after unzip: {missing}\nRe-zip so these paths sit under data/ (zip the CREMA-D / MELD folder itself).'
for p,_ in required:
    full=os.path.join(DST,p)
    n=len(os.listdir(full)) if os.path.isdir(full) else 1
    print(f'  OK  {p}  ({n} entries)')
print('layout verified.')

## 3c. VERIFY data content + split

In [ ]:
# data-content verification: resolve samples + check the split is non-empty
import sys; sys.path.insert(0,'/content/Baseline_Training')
from src.config import TrainConfig
from src.data import manifests
from src.data.splits import partition_by_actor
from src.train_utils import stratified_split
cfg=TrainConfig()
samples=manifests.build_emotion_samples('crema')
assert len(samples)>0, 'NO SAMPLES RESOLVED -- check zip structure / file naming'
tr,va,te=partition_by_actor(samples, lambda s:s.group_key,(0.8,0.1,0.1),cfg.seed)
assert min(len(tr),len(va),len(te))>0, 'a split partition is empty'
from collections import Counter
names=cfg.EMOTION_DATASETS['crema']['emotions'] if hasattr(cfg,'EMOTION_DATASETS') else None
print(f'resolved {len(samples)} samples  ->  train {len(tr)} / val {len(va)} / test {len(te)}')
print('label counts:', dict(Counter(s.emotion for s in samples)))
print('VERIFIED: data present and split non-empty.')

## 4. Train (50 epochs, patience 25, augmentation on)

In [ ]:
!cd /content/Baseline_Training && PYTHONPATH=. python -m src.train_stage1 --branch speech_text --dataset crema --batch-size 32 --epochs 50 --early-stop 25 --num-workers 2

## 5. Back up checkpoint to Drive (Colab is ephemeral!)

In [ ]:
import os, shutil
os.makedirs('/content/drive/MyDrive/acenet_ckpts', exist_ok=True)
for f in ['stage1_speech_text_crema.pt','stage1_speech_text_crema.log']:
    src=f'/content/Baseline_Training/checkpoints/{f}'
    if os.path.exists(src): shutil.copy(src, f'/content/drive/MyDrive/acenet_ckpts/{f}')
print('backed up:', os.listdir('/content/drive/MyDrive/acenet_ckpts'))

## 6. Evaluate (Accuracy, Weighted-F1, per-class F1, confusion)

In [ ]:
!cd /content/Baseline_Training && PYTHONPATH=. python -m src.eval_stage1 --branch speech_text --dataset crema

## Hand-off

`stage1_speech_text_crema.pt` is now on Drive in `acenet_ckpts/`.
**Send the two CREMA checkpoints (`stage1_visual_crema.pt`, `stage1_speech_text_crema.pt`) to the Stage-2 runner.**